In [1]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

In [2]:
import keras

In [3]:
df=pd.read_csv(r"..\data\pixels.csv")

In [4]:
df.shape

(13770, 3073)

In [5]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,3063,3064,3065,3066,3067,3068,3069,3070,3071,target
0,121,133,137,131,141,135,143,147,138,146,...,149,145,145,149,144,144,119,120,137,1
1,255,255,255,0,0,0,0,0,0,0,...,255,255,255,255,255,255,255,255,255,1
2,31,30,225,34,35,218,28,41,207,20,...,23,35,220,24,40,210,28,36,216,1
3,0,0,0,0,0,0,255,255,255,255,...,255,255,255,255,255,255,0,0,0,1
4,203,200,237,234,230,240,255,255,248,251,...,123,121,224,3,11,203,27,35,223,1


In [6]:
df['target'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52,
       53, 54, 55, 56, 57, 58])

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X=df.drop(columns=['target'])
y=df['target']

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,shuffle=True,test_size=0.2, stratify=y)

In [10]:
X_train.head()

,0,1,2,3,4,5,6,7,8,9,...,3062,3063,3064,3065,3066,3067,3068,3069,3070,3071
3891,176,173,244,251,247,252,244,252,250,246,...,217,28,39,216,22,38,213,20,25,224
11473,160,143,129,150,128,110,148,123,104,145,...,43,120,82,48,122,84,52,122,86,54
2496,255,255,255,255,255,255,255,255,255,255,...,255,255,255,255,255,255,255,255,255,255
6199,255,255,255,255,255,255,255,255,255,255,...,255,8,8,255,20,20,255,120,120,255
2677,255,255,255,255,255,255,255,255,255,255,...,255,255,255,255,255,255,255,255,255,255


In [17]:
df['target'].nunique()

57

In [26]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [13]:
import numpy as np
import optuna

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [27]:
def objective(trial):

    # Number of hidden layers
    n_layers = trial.suggest_int("n_layers", 1, 4)

    model = Sequential()

    model.add(Dense(trial.suggest_int("neurons_layer_0",32,256,step=32),
            activation="relu",
            input_shape=(X_train.shape[1],)
        )
    )

    # Additional hidden layers
    for i in range(1, n_layers):

        neurons = trial.suggest_int(
            f"neurons_layer_{i}",
            32,
            256,
            step=32
        )

        model.add(
            Dense(
                neurons,
                activation="relu"
            )
        )

        dropout = trial.suggest_float(
            f"dropout_{i}",
            0.0,
            0.5,
            step=0.1
        )

        model.add(Dropout(dropout))

    # Output layer
    model.add(
        Dense(
            59,
            activation="softmax"
        )
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64, 128]
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=30,
        batch_size=batch_size,
        verbose=0
    )

    return max(history.history["val_accuracy"])

In [28]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

[I 2026-09-01 12:14:46,877] A new study created in memory with name: no-name-73fda2a2-09e4-4762-b450-e013395b4985
d:\Data Science\DataScienceProject\vehicle_sign_prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-09-01 12:15:14,356] Trial 0 finished with value: 0.48865699768066406 and parameters: {'n_layers': 3, 'neurons_layer_0': 32, 'neurons_layer_1': 64, 'dropout_1': 0.30000000000000004, 'neurons_layer_2': 96, 'dropout_2': 0.5, 'learning_rate': 0.00011158735778822887, 'batch_size': 64}. Best is trial 0 with value: 0.48865699768066406.
d:\Data Science\DataScienceProject\vehicle_sign_prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` a

In [29]:
print("Best validation accuracy:")
print(study.best_value)

print("\nBest hyperparameters:")
print(study.best_params)

Best validation accuracy:
0.754083514213562

Best hyperparameters:
{'n_layers': 3, 'neurons_layer_0': 160, 'neurons_layer_1': 192, 'dropout_1': 0.1, 'neurons_layer_2': 160, 'dropout_2': 0.30000000000000004, 'learning_rate': 0.0003213819761880707, 'batch_size': 16}


In [30]:
best = study.best_params

model = Sequential()

model.add(
    Dense(
        best["neurons_layer_0"],
        activation="relu",
        input_shape=(X_train.shape[1],)
    )
)

for i in range(1, best["n_layers"]):

    model.add(
        Dense(
            best[f"neurons_layer_{i}"],
            activation="relu"
        )
    )

    model.add(
        Dropout(best[f"dropout_{i}"])
    )

model.add(
    Dense(
        59,
        activation="softmax"
    )
)

d:\Data Science\DataScienceProject\vehicle_sign_prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=best["learning_rate"]
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [32]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [33]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=best["batch_size"],
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.1651 - loss: 3.3855 - val_accuracy: 0.2813 - val_loss: 2.6002
Epoch 2/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.3339 - loss: 2.4397 - val_accuracy: 0.4519 - val_loss: 1.9772
Epoch 3/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4181 - loss: 2.0400 - val_accuracy: 0.5009 - val_loss: 1.7743
Epoch 4/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4881 - loss: 1.7713 - val_accuracy: 0.5740 - val_loss: 1.5281
Epoch 5/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5355 - loss: 1.5892 - val_accuracy: 0.5971 - val_loss: 1.4158
Epoch 6/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5743 - loss: 1.4286 - val_accuracy: 0.6216 - val_loss: 1.3016
Epoch 7/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6119 - loss: 1.3034 - val_accuracy: 0.6461 - val_loss: 1.2323
Epoch 8/100
551/551 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6429 - loss: 1.1604 - val_accu

In [34]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7393 - loss: 0.9852
Test Loss: 0.9852256774902344
Test Accuracy: 0.739288330078125


In [35]:
y_pred_prob = model.predict(X_test)

87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [36]:
y_pred = np.argmax(y_pred_prob, axis=1)

In [37]:
y_pred

array([36, 55, 51, ..., 22, 55, 48], shape=(2754,))

In [38]:
print("Actual:   ", y_test[:20])
print("Predicted:", y_pred[:20])

Actual:    8237     37
12742    55
10593    51
5097     21
10141    48
9701     45
11110    51
11253    52
8535     38
6657     29
2735     12
2551     11
11066    51
3884     17
6724     29
9128     42
6704     29
9101     42
1437      7
10265    48
Name: target, dtype: int64
Predicted: [36 55 51 21 48 45 51 52 40 12 12 11 51 17 29 42 29 42  7 48]


In [39]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)
print("Test Accuracy %:", accuracy * 100)

Test Accuracy: 0.739288307915759
Test Accuracy %: 73.92883079157589


In [40]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.86      0.75      0.80        40
           2       0.52      0.30      0.38        40
           3       0.54      0.63      0.58        41
           4       0.86      0.90      0.88        40
           5       0.77      0.77      0.77        64
           6       0.91      0.72      0.81        40
           7       0.68      0.85      0.76        40
           8       0.85      0.83      0.84       120
           9       0.88      0.70      0.78        40
          10       0.89      0.83      0.86        41
          11       0.86      0.90      0.88        41
          12       0.69      0.82      0.75        40
          13       0.83      0.85      0.84        40
          14       0.83      0.75      0.79        40
          15       0.61      0.68      0.64        40
          16       0.63      0.65      0.64        40
          17       0.84      0.80      0.82        40
          18       0.84    

In [41]:
model.save("vehicle_sign_ann.keras")

In [43]:

df_signs = pd.read_csv(
    r"D:\Data Science\DataScienceProject\vehicle_sign_prediction\data\traffic_sign.csv"
)

In [44]:
sign_mapping = dict(
    zip(df_signs["ClassId"], df_signs["Name"])
)

In [45]:
prediction = y_pred[0]

print("Predicted Class ID:", prediction)
print("Predicted Sign:", sign_mapping[prediction])

Predicted Class ID: 36
Predicted Sign: Side road junction
